In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, KFold
from sksurv.util import Surv
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import concordance_index_censored
from sksurv.nonparametric import kaplan_meier_estimator

In [2]:
seed = 42
time_for_eval = 24
quick = True

R, K = (1, 5) if quick else (5, 5)
n_trees = 200 if quick else 500

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

data = pd.read_csv("data/acrg_clean.csv")
features = ["age", "male", "serosa", "n_level"]
data["male"] = (data["sex"] == "Male").astype(float)
X = data[features].astype(float)
y = Surv.from_arrays(event=data["event"].astype(bool).values, time=data["time_months"].values)
print(X.shape, "events:", int(data["event"].sum()))

(252, 4) events: 103


In [3]:
def c_index_time_for_eval(y_true, risk, tfe = time_for_eval):
    event_tfe = y_true["event"] & (y_true["time"] < tfe)
    return concordance_index_censored(event_tfe, y_true["time"], risk)[0]

def eval_times(y_test, tfe = time_for_eval):
    hi = min(tfe, y_test["time"].max() - 1e-6)
    lo = max(1.0, y_test["time"].min() + 1e-6)
    return np.linspace(lo, hi, 24)

def surv_at(model, X_, times):
    return np.vstack([fn(times) for fn in model.predict_survival_function(X_)])

def km_null_surv(y_train, times, n):
    t, s = kaplan_meier_estimator(y_train["event"], y_train["time"])
    s_at = np.array([s[t <= u][-1] if (t <= u).any() else 1.0 for u in times])
    return np.tile(s_at, (n, 1))

In [4]:
def make_cox():
    pipe = Pipeline([("scale", StandardScaler()),("cox", CoxnetSurvivalAnalysis(fit_baseline_model = True))])
    grid = {"cox_alphas": [[1e-3], [1e-2], [1e-1], [1.0]], "cox_l1_ratio": [1.0, 0.5, 0.9]}
    return GridSearchCV(pipe, grid, cv=KFold(3, shuffle=True, random_state=seed), error_score = 0.5)

def make_rfs():
    rsf = RandomSurvivalForest(n_estimators=n_trees, random_state=seed, n_jobs=-1)
    grid = {"min_samples_leaf": [5, 10, 20], "max_features": ["sqrt, 0.5, 1.0"]}
    return GridSearchCV(rsf, grid, cv=KFold(3, shuffle=True, random_state=seed))

models = {"cox": make_cox(), "rfs": make_rfs()}